<a href="https://colab.research.google.com/github/Riana-PSB/Assignment-1/blob/CNN-Model/cnn_v3_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, we will be creating a CNN (Convolutional Neural Network) based classifier for the classification of CIFAR-10 dataset.

**About the dataset**

CIFAR-10 dataset is a subset of CIFAR-100 having 80 million images of 100 different objects. CIFAR-10 takes 60,000 images for 10 classes from the original dataset in which 50,000 images are for training and 10,000 images are for testing.

The classes in the dataset are -
* airplane
* automobile
* bird
* cat
* deer
* dog
* frog
* horse
* ship
* truck

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    # The CIFAR labels happen to be arrays,
    # which is why you need the extra index
    plt.xlabel(class_names[train_labels[i][0]])
plt.show()

**Creating a CNN model**

We are going to create a CNN based classification model using *keras* module.

In [ ]:
model = models.Sequential()

model.add(layers.Conv2D(32,(3,3),activation='relu', input_shape=(32,32,3)))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.Conv2D(64,(3,3),activation='relu'))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.Conv2D(64,(3,3),activation='relu'))

model.summary()

In [ ]:
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10))

model.summary()

In [ ]:
model.compile(optimizer='sgd',loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
history = model.fit(train_images, train_labels, epochs=10,validation_data=(test_images, test_labels))


In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')
test_loss, test_acc = model.evaluate(test_images,  test_labels, verbose=2)

In [ ]:
print(test_acc)

In [ ]:
model.compile(optimizer='adam',loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
history = model.fit(train_images, train_labels, epochs=10,validation_data=(test_images, test_labels))

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')
test_loss, test_acc = model.evaluate(test_images,  test_labels, verbose=2)

In [ ]:
print(test_acc)

In [ ]:
# save model
model.save('final_model.keras')

In [ ]:
from tensorflow.keras import preprocessing, models
import numpy as np
import cv2

# CIFAR-10 class names
class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

# --- Load + preprocess image ---
def load_image(filename):
    img = preprocessing.image.load_img(filename)
    img = preprocessing.image.img_to_array(img)

    # Resize FIRST
    img = cv2.resize(img, (32, 32))

    # Stronger blur (important!)
    img = cv2.GaussianBlur(img, (5, 5), 0)

    # Reduce detail further (simulate CIFAR)
    img = cv2.GaussianBlur(img, (3, 3), 0)

    # Normalize
    img = img.astype('float32') / 255.0

    return img

# --- Run predictions on multiple images ---
def run_multiple():
    # List of images (3 normal + 1 Webots)
    img_paths = [
    "/content/Cat.jpg",
    "/content/Ship.jpg",
    "/content/Frog.jpg",
    "/content/cat_capture_0.png"
]

    # Load model once
    model = models.load_model('final_model.keras')

    processed_images = []

    for path in img_paths:
        try:
            img = load_image(path)
            processed_images.append(img)
        except Exception as e:
            print(f"Error loading {path}: {e}")

    # Convert to numpy batch
    processed_images = np.array(processed_images)

    # Predict
    predictions = model.predict(processed_images)

    # Print results
    for i, pred in enumerate(predictions):
        result = np.argmax(pred)
        confidence = np.max(pred)

        print(f"{img_paths[i]} → {class_names[result]} ({confidence:.2f})")

# Run
run_multiple()